# Lab 09-01 — Chunk embeddings + GMM clustering (RAPTOR step 1)

**Track 09 · RAPTOR** — replacing flat chunk lists with a summary tree.

RAPTOR (Sarthi et al., 2024) builds a hierarchical index over a corpus. The first half of that index is **unsupervised**: embed every chunk and group the embeddings so that related chunks land in the same cluster. Each cluster is small enough to be compressed by a summary without losing the details — the summarization happens in lab 02, so this lab never calls an LLM.

This notebook is **self-contained**: it imports LangChain (for the BGE embedder), pandas, and scikit-learn directly — no repo component library. The recursive Gaussian-mixture clustering that the repo ships as `src/tools/raptor.py` is hand-rolled right here, cell by cell, which is exactly how that shared component works underneath.

```text
30 passages (rag-mini-wikipedia, deterministic head)
  -> BGE embeddings (BAAI/bge-base-en-v1.5, local, CPU)
  -> recursive GaussianMixture (2-component, split while > max_cluster_size)
  -> partition of clusters (every chunk appears exactly once)
  -> verification gate
```

The clustering recipe from the paper: fit a 2-component `GaussianMixture` over the current chunk embeddings; if a resulting cluster is at most `max_cluster_size` chunks, keep it as a leaf cluster; otherwise split that cluster again (another GMM inside it). The output is a *partition* — every chunk index appears in exactly one cluster — and that partition is the raw material for the tree: lab 02 turns each cluster into a summary node.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet` (3200 passages), already fetched by the repo's manifest-verified fetchers.

There is **no Ollama prerequisite** here: this lab never calls an LLM, so it runs fully offline. No repo imports are needed either: everything comes from `langchain-huggingface`, `pandas`, `numpy`, and `scikit-learn`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   langchain-huggingface -> HuggingFaceEmbeddings (BGE backend)
#   pandas                -> read the rag-mini-wikipedia parquet
#   scikit-learn, numpy   -> GaussianMixture clustering (hand-rolled RAPTOR)
%pip install -q langchain-huggingface pandas scikit-learn


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# LangChain + pandas + scikit-learn — the only libraries this notebook
# needs. Nothing is imported from the repo's src/ component library.
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from sklearn.mixture import GaussianMixture  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `PASSAGES_PATH` points at the rag-mini-wikipedia corpus already on disk; `N_PASSAGES = 30` takes a deterministic head of the 3200-passage corpus (no randomness, reproducible runs); `MAX_CLUSTER_SIZE = 10` is the cap that decides when a cluster is small enough to be a leaf; `SEED = 42` makes the GMM splits reproducible; `BGE_MODEL_NAME` / `BGE_DEVICE` pin the embedder to the local BGE model on CPU (Ollama holds most of the shared GPU's VRAM).


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
N_PASSAGES = 30  # deterministic head; embedding + clustering only, no LLM calls
MAX_CLUSTER_SIZE = 10  # clusters larger than this get split recursively
SEED = 42  # deterministic GMM splits
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM


## 2. Load — first N passages of the rag-mini-wikipedia corpus

`load_passages` reads the first `n` rows of `passages.parquet` and returns the stripped passage texts in file order — a deterministic head, exactly the slice the lab script uses.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N passages of the rag-mini-wikipedia corpus
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic head)."""
    df = pd.read_parquet(path)
    return [str(text).strip() for text in df.head(n)["passage"].tolist()]


## 3. Experiment — embed the chunks, then cluster with the recursive GMM

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model — `model_kwargs={"device": "cpu"}` and `normalize_embeddings=True`, which BGE requires for cosine — and we embed the 30 passages once. Then `cluster_embeddings` hand-rolls the RAPTOR recipe from the paper: `_split_indices` fits a 2-component full-covariance `GaussianMixture` over the current index set; a cluster at most `max_cluster_size` is kept as a leaf; anything larger is split again. When the fit fails (too few points, degenerate covariance, or a collapsed component) the whole set is returned as one cluster, so the partition always covers every index exactly once. This is the same algorithm the shared `src/tools/raptor.py` function implements.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed the chunks, then cluster them with the recursive GMM
# --------------------------------------------------------------------------
def _split_indices(
    index_set: list[int],
    embeddings: list[list[float]],
    max_cluster_size: int,
    seed: int,
) -> list[list[int]]:
    """Recursively split ``index_set`` into clusters of at most ``max_cluster_size``.

    Fits a 2-component full-covariance GaussianMixture on the current index
    set. A cluster small enough is kept as a leaf; anything larger is split
    again. When the fit fails (too few points, degenerate covariance, or a
    collapsed component) the whole set is returned as one cluster, so the
    partition always covers every index exactly once.
    """
    if len(index_set) <= max_cluster_size:
        return [sorted(index_set)]
    matrix = np.asarray([embeddings[i] for i in index_set], dtype=float)
    try:
        labels = GaussianMixture(
            n_components=2,
            covariance_type="full",
            random_state=seed,
        ).fit_predict(matrix)
    except (ValueError, np.linalg.LinAlgError):
        # Fallback: keep the whole set as one cluster on fit failure
        # (ill-defined covariance, n_components > n_samples) so coverage holds.
        return [sorted(index_set)]

    groups: dict[int, list[int]] = {}
    for idx, label in zip(index_set, labels):
        groups.setdefault(int(label), []).append(idx)
    if len(groups) < 2:  # degenerate split: every point in one component
        return [sorted(index_set)]

    return [
        cluster
        for group in groups.values()
        for cluster in _split_indices(group, embeddings, max_cluster_size, seed)
    ]


def cluster_embeddings(
    embeddings: list[list[float]],
    max_cluster_size: int,
    seed: int = 42,
) -> list[list[int]]:
    """Partition chunk indices into GMM clusters of at most ``max_cluster_size``.

    Every index in ``range(len(embeddings))`` appears in exactly one cluster
    (the recursion is over a partition of the index set). Returns a single
    cluster when fitting fails or there are too few points.
    """
    if not embeddings:
        return []
    return _split_indices(
        list(range(len(embeddings))), embeddings, max_cluster_size, seed
    )


def run_experiment() -> dict:
    passages = load_passages(PASSAGES_PATH, N_PASSAGES)
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": BGE_DEVICE},
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )

    t0 = time.perf_counter()
    embeddings = embedder.embed_documents(passages)
    embed_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    clusters = cluster_embeddings(
        embeddings, max_cluster_size=MAX_CLUSTER_SIZE, seed=SEED
    )
    cluster_s = time.perf_counter() - t0

    return {
        "passages": passages,
        "embeddings": embeddings,
        "clusters": clusters,
        "embed_s": embed_s,
        "cluster_s": cluster_s,
    }


## 4. Demo — print the partition

`print_demo(exp)` prints the artifact from three angles: the corpus subset with embedding/clustering timings; the cluster sizes (largest first) proving the `max_cluster_size` cap held; one sample passage per cluster so you can eyeball whether related chunks actually landed together; then a takeaway explaining why the partition is the raw material for the tree.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the partition
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 09-01 — Chunk embeddings + GMM clustering (RAPTOR step 1)")
    n = len(exp["passages"])
    clusters = exp["clusters"]
    print(f"{n} passages embedded in {exp['embed_s']:.1f}s; "
          f"{len(clusters)} clusters (cap {MAX_CLUSTER_SIZE}) found in "
          f"{exp['cluster_s']:.1f}s")
    print("=" * 66)

    sizes = sorted((len(c) for c in clusters), reverse=True)
    print(f"\n[1] Cluster sizes (largest first): {sizes}")

    print(f"\n[2] One sample passage per cluster (first 120 chars):")
    for i, cluster in enumerate(clusters):
        sample = exp["passages"][cluster[0]]
        print(f"    C{i} (size {len(cluster)}): {sample[:120]}")

    print(f"\n[3] Takeaway")
    print("    Clustering is the unsupervised half of RAPTOR's index: each")
    print("    cluster groups chunks that a single summary can compress.")
    print("    The partition covers every chunk exactly once, so lab 02 can")
    print("    build one summary node per cluster without losing chunks.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: every chunk index assigned exactly once (the partition covers `range(N)`), the cluster count in a sane band `[2, N//2]`, every cluster non-empty, and every cluster at most `max_cluster_size`. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    clusters = exp["clusters"]
    n = len(exp["passages"])
    flat = sorted(i for cluster in clusters for i in cluster)

    checks.append(("every chunk index assigned exactly once "
                   f"(union == set(range({n})))",
                   flat == list(range(n))))
    checks.append((f"cluster count in [2, {n // 2}] (got {len(clusters)})",
                   2 <= len(clusters) <= n // 2))
    checks.append(("every cluster is non-empty",
                   all(len(cluster) > 0 for cluster in clusters)))
    checks.append((f"every cluster size <= {MAX_CLUSTER_SIZE}",
                   all(len(cluster) <= MAX_CLUSTER_SIZE for cluster in clusters)))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Under a minute of embedding + a few GMM fits on 30 passages — no downloads, no API calls, no LLM. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The partition itself: cluster sizes (largest first) and one sample passage per cluster, so you can see that related chunks landed together — the raw material lab 02 turns into summary nodes.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet file is intact.


In [ ]:
verify_gate(exp)
